# CSV (2026년 최신 권장 사용법)

[Comma-Separated Values (CSV)](https://en.wikipedia.org/wiki/Comma-separated_values) 파일은 쉼표로 값을 구분하는 구분된 텍스트 파일입니다. 파일의 각 줄은 데이터 레코드입니다.

각 레코드는 쉼표로 구분된 하나 이상의 필드로 구성됩니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `CSVLoader` | 표준 라이브러리 `csv` 로 행을 읽어 `Document` 로 변환하는 작은 로더 직접 작성 |
| `UnstructuredCSVLoader(mode="elements")` | **`langchain-unstructured`** 의 `UnstructuredLoader` |
| `DataFrameLoader` | `pandas` 로 직접 `Document` 생성 |

In [ ]:
# 설치
# !pip install -qU langchain-core pandas
# !pip install -qU langchain-unstructured "unstructured[csv]"   # UnstructuredLoader 사용 시

## CSV 행 → Document 로더 (구 `CSVLoader`)

- CSV 데이터를 **문서당 한 행씩** 로드합니다.
- `page_content` 는 책의 `CSVLoader` 와 같은 `"컬럼: 값"` 줄 형식입니다.
- `metadata` 에는 `source`(파일 경로 또는 `source_column` 값) 와 `row`(행 번호) 가 들어갑니다.

In [ ]:
import csv
from pathlib import Path
from typing import Any, Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


class CSVRowLoader(BaseLoader):
    """CSV 의 각 행을 Document 로 변환하는 로더"""

    def __init__(
        self,
        file_path: str | Path,
        *,
        source_column: str | None = None,
        metadata_columns: list[str] | None = None,
        csv_args: dict[str, Any] | None = None,
        encoding: str = "utf-8",
    ) -> None:
        self.file_path = Path(file_path)
        self.source_column = source_column
        self.metadata_columns = metadata_columns or []
        self.csv_args = csv_args or {}
        self.encoding = encoding

    def lazy_load(self) -> Iterator[Document]:
        with self.file_path.open(newline="", encoding=self.encoding) as f:
            reader = csv.DictReader(f, **self.csv_args)
            for i, row in enumerate(reader):
                content = "\n".join(
                    f"{k.strip()}: {(v or '').strip()}"
                    for k, v in row.items()
                    if k is not None and k not in self.metadata_columns
                )
                source = row[self.source_column] if self.source_column else str(self.file_path)
                metadata = {"source": source, "row": i}
                metadata.update({c: row.get(c) for c in self.metadata_columns})
                yield Document(page_content=content, metadata=metadata)

In [ ]:
# CSV 로더 생성
loader = CSVRowLoader("./data/titanic.csv")

# 데이터 로드
docs = loader.load()

print(len(docs))
print(docs[0].metadata)

In [ ]:
print(docs[1].page_content)

### CSV 파싱 및 로딩 커스터마이징

`csv_args` 는 그대로 `csv.DictReader` 에 전달됩니다. 지원 인자는 [csv module](https://docs.python.org/3/library/csv.html) 문서를 참고하세요.

> `fieldnames` 를 직접 지정하면 파일의 **첫 줄(원래 헤더)도 데이터 행으로 읽힙니다.** 그래서 아래에서 `docs[0]` 은 헤더 행이 됩니다. (책과 동일한 동작)

In [ ]:
# 컬럼정보:
# PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked

loader = CSVRowLoader(
    "./data/titanic.csv",
    csv_args={
        "delimiter": ",",  # 구분자
        "quotechar": '"',  # 인용 부호 문자
        "fieldnames": [
            "Passenger ID",
            "Survival (1: Survived, 0: Died)",
            "Passenger Class",
            "Name",
            "Sex",
            "Age",
            "Number of Siblings/Spouses Aboard",
            "Number of Parents/Children Aboard",
            "Ticket Number",
            "Fare",
            "Cabin",
            "Port of Embarkation",
        ],  # 필드 이름
    },
)

# 데이터 로드
docs = loader.load()

# 데이터 출력
print(docs[1].page_content)

### 행을 XML 형식으로 변환

책에서는 `page_content` 문자열을 `:` 로 다시 쪼개서 XML 을 만들었습니다. 하지만 값에 `:` 가 있거나 컬럼명에 공백·괄호가 있으면 **유효하지 않은 XML** 이 됩니다.

원본 행(dict)에서 바로 만들고, 태그명과 값을 안전하게 처리하는 것이 좋습니다.

In [ ]:
import re
from xml.sax.saxutils import escape


def to_tag(name: str) -> str:
    """XML 태그로 쓸 수 있도록 컬럼명을 정리"""
    tag = re.sub(r"\W+", "_", name.strip()).strip("_")
    return tag if tag and not tag[0].isdigit() else f"col_{tag}"


def row_to_xml(row: dict[str, str]) -> str:
    inner = "".join(
        f"<{to_tag(k)}>{escape((v or '').strip())}</{to_tag(k)}>" for k, v in row.items()
    )
    return f"<row>{inner}</row>"


with open("./data/titanic.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))  # 헤더를 자동 인식하므로 스킵 불필요

print(row_to_xml(rows[0]))

In [ ]:
# XML 을 page_content 로 가진 Document 목록
xml_docs = [
    Document(page_content=row_to_xml(row), metadata={"source": "./data/titanic.csv", "row": i})
    for i, row in enumerate(rows)
]
for d in xml_docs[:5]:
    print(d.page_content)

`source_column` 인자를 사용하여 각 행에서 생성된 문서의 출처를 지정할 수 있습니다. 지정하지 않으면 모든 문서의 출처로 파일 경로가 사용됩니다.

이는 CSV 파일에서 로드된 문서를 출처를 사용하여 질문에 답하는 체인에 사용할 때 유용합니다.

In [ ]:
loader = CSVRowLoader(
    "./data/titanic.csv", source_column="PassengerId"
)  # CSV 로더 설정, 파일 경로 및 소스 컬럼 지정

docs = loader.load()  # 데이터 로드

print(docs[1])  # 데이터 출력

## UnstructuredLoader (구 `UnstructuredCSVLoader`)

`langchain-unstructured` 의 `UnstructuredLoader` 로 CSV 를 표(Table) 요소로 로드할 수 있습니다. 표 요소의 메타데이터 `text_as_html` 에 **테이블의 HTML 표현**이 제공됩니다. (별도의 `mode="elements"` 지정 불필요)

In [ ]:
from langchain_unstructured import UnstructuredLoader

# Unstructured 로더 인스턴스 생성 (파일 형식 자동 감지)
loader = UnstructuredLoader("./data/titanic.csv")

# 문서 로드
docs = loader.load()

# 첫 번째 문서의 HTML 텍스트 메타데이터 출력
print(docs[0].metadata["text_as_html"][:1000])

## pandas DataFrame → Document (구 `DataFrameLoader`)

- Pandas 는 Python 을 위한 오픈 소스 데이터 분석 및 조작 도구입니다.
- `DataFrameLoader` 가 하던 일(한 컬럼은 `page_content`, 나머지 컬럼은 `metadata`)을 짧은 함수로 대체합니다.

In [ ]:
import pandas as pd

# CSV 파일 읽기
df = pd.read_csv("./data/titanic.csv")

첫 5개 행을 조회합니다.

In [ ]:
# 데이터프레임의 처음 다섯 행 조회
df.head()

In [ ]:
from typing import Iterator


def dataframe_to_documents(df: pd.DataFrame, page_content_column: str) -> Iterator[Document]:
    """한 컬럼은 본문, 나머지 컬럼은 메타데이터로 변환 (generator = 지연 로딩)"""
    # NaN 은 JSON 직렬화/벡터스토어 저장 시 문제가 되므로 None 으로 변환
    meta_df = df.drop(columns=[page_content_column]).astype(object)
    meta_df = meta_df.where(pd.notna(meta_df), None)
    for content, metadata in zip(df[page_content_column], meta_df.to_dict(orient="records")):
        yield Document(page_content=str(content), metadata=metadata)


# 문서 로드
docs = list(dataframe_to_documents(df, page_content_column="Name"))

# 데이터 출력
print(docs[0].page_content)

# 메타데이터 출력
print(docs[0].metadata)

In [ ]:
# 큰 테이블에 대한 지연 로딩: generator 이므로 필요한 만큼만 Document 를 생성
for row in dataframe_to_documents(df, page_content_column="Name"):
    print(row)
    break  # 첫 행만 출력

> 💡 **행 전체를 본문으로 쓰고 싶다면** `page_content` 를 `"컬럼: 값"` 줄 형식(위 `CSVRowLoader` 와 동일)이나 JSON 문자열로 만드는 것이 검색 품질에 더 유리한 경우가 많습니다.
>
> ```python
> docs = [
>     Document(page_content=json.dumps(rec, ensure_ascii=False), metadata={"row": i})
>     for i, rec in enumerate(df.to_dict(orient="records"))
> ]
> ```